# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |
| [16-security-review-2026-09-25](16-security-review-2026-09-25.ipynb) | **診断の報告書(2026-09-25)。** 前回(10)の未確認範囲・ログイン・個人情報の扱い。所見 W-44〜W-52(Android は別の報告書) |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()


# 16. 診断の報告(2026-09-25)— Website

**添付の脆弱性カタログ(`security-review-vulnerability-catalog.md`、2026-09-13 付)を物差しにして、
前回の診断([10](10-security-review-2026-09-14.ipynb))で「読んでいない」とした範囲と、その後に入った機能を調べた。**
重点は2つ: **ログイン**と**個人情報がありうる場所**。

**この報告は Website(server/)の分だけ。** Android アプリの所見(A-29〜A-33)は、非公開の Android リポジトリ(`itotakusub/Ichinoseki_Kosen`)の `docs/security-review-2026-09-25.ipynb` に分けた。

> **この報告では何も直していない。** 所見と直し方の案だけ。直すかどうか・どの順かは利用者が決める(§5)。
> **本番には一切触れていない**(SSH・curl・配備のどれもしていない)。
> **全所見を、手元に組んだ検証環境で実際に動かして確かめた(§7)。** 検証の途中で見つけた W-53 も足した。

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。手元のファイルを読んで所見を確かめるセル |

**個人情報をこの文書に書き写していない。** IP アドレス・メールアドレスなどは「どのファイルの何行目にあるか」だけ書いた
(このリポジトリは公開されているので、ここに値を書くと広げることになる)。

## 0. この報告の読み方

### 0.1 目次

| 節 | 中身 |
|---|---|
| §1 | **要約。** 件数・まず読む3つ・問題が見つからなかったもの |
| §2 | **何を調べたか。** 前回の未確認範囲をどう埋めたか・カタログの節ごとの結果 |
| §3 | **所見の一覧**(W-44〜W-52。番号は前回 10 の続き) |
| §4 | **所見の詳細と、確かめるセル** |
| §5 | **直し方の案(利用者の判断待ち)** |
| §6 | **この診断の限界** |
| §7 | **検証。** 検証環境・本番との違い・所見ごとの実行結果・やり直し方 |

### 0.2 確かさの区別

| 判定 | 意味 |
|---|---|
| **実行で確認** | 該当するコードを手元の検証環境で実際に動かして、書いたとおりに振る舞うことを確かめた(§7)。**本番では確かめていない** |

最初の版(同じ日の朝)では「コードで確認」「可能性」だったものも、**すべて実行で確認できた。** 反証されたものは無い。

深刻度は**カタログの区分(重大・高・中・低・情報)を、この環境で届く範囲と影響に合わせて付け直した値**。
**「重大」「高」は無い。** 「中」が 2 件。

**file:line は 2026-09-25 の作業ツリー(`main` と同じ)の行番号。**

## 1. 要約

### 1.1 件数

**所見 10 件、すべて実行で確認。** 重大・高は 0 件。

| 深刻度 | 件数 | 番号 |
|---|---|---|
| 中 | **2** | W-44・W-53 |
| 低 | **6** | W-45・W-46・W-47・W-48・W-49・W-50 |
| 情報 | **2** | W-51・W-52 |

**種類で分けると:** セキュリティ 3 件・個人情報の扱い 6 件・不具合(予期しない動作)1 件。

### 1.2 まず読む3つ

| # | 何か | 番号 | 深刻度 |
|---|---|---|---|
| 0 | **公開リポジトリに、実在の教職員の氏名と部屋の組が 4 人分ある。** テストのデータ・コメント・計画書に残っている。地図のパスワードで隠している情報そのもの(検証の途中で見つけた) | W-53 | 中 |
| 1 | **ランキングの記録が、同じ地点の重複と実在しない地点を数える。** ログインした参加者は 1 回の送信で自分の件数を +50 でき、誰でも実在しない ID・新しい語で表の行を際限なく増やせる(1 つの IP から毎分最大 120 回) | W-44 | 中 |
| 2 | **公開リポジトリに運用者の回線の IP アドレスが載っている。** 管理用ポートへの接続と `kmops` の鍵の使用を許している回線。push 前の検査は IP を見ていない | W-47 | 低 |
| 3 | **アカウントを削除しても、対象者の利用者 ID が監査ログとチャットに残る。** プライバシーポリシー §8 の「利用者 ID は削除する」と食い違う | W-48 | 低 |

### 1.3 問題が見つからなかったもの(今回新たに確かめた範囲)

- **ログインの流れ(Logto PHP SDK 0.3.1 の中身を取り寄せて読んだ)**: PKCE・`state` の照合・ID トークンの署名/`iss`/`aud` の検証がある。コールバックの戻り先は同一サイト内に限られ、`$appUrl` を前置している(オープンリダイレクト無し)
- **依存ライブラリの既知の脆弱性**: `composer.lock` どおりに取り寄せて `composer audit --locked` → **"No security vulnerability advisories found."**(2026-09-25 時点の Packagist の公開情報)
- **Logto `1.43.0`・phpMyAdmin `5.2.3`**: カタログが修正版の下限とする Logto 1.41.0・phpMyAdmin 5.2.2 以上で、どちらも digest で固定(`compose.yaml:112` `:243`)
- **Soketi**: 購読の認可は管理者だけ・許可リストの presence チャネルだけ(`admin/api/chat-auth.php`)。クライアントからの送信は無効(`compose.yaml:650`)
- **死活監視(`admin/api/health-check.php`)**: 問い合わせ先は固定で、利用者が選べない(SSRF 無し)
- **地図データの HTML への埋め込み**: `index.php:127` と `admin/map-editor.php:56` はどちらも `JSON_HEX_TAG` 付き。教職員が提案した名前が承認されても `</script>` は作れない
- **管理画面のチャットの表示**: 表示前にエスケープしている(`admin/chat.php:211`)
- **教職員の申請・担当地点の提案**: 状態の遷移は条件付き UPDATE で二重処理されない。承認の時点で「割り当て」と「申請が承認済みか」をもう一度見る(`lib/staff-nodes.php` の `km_staff_node_edit_decide`)
- **このリポジトリの git 履歴の秘密情報**: 全コミットの追加行を、パスワード・鍵・トークンの形で検索して 0 件

## 2. 何を調べたか

### 2.1 前回(10 §8)の「読んでいないコード」をどう埋めたか

| 前回の未確認 | 今回 |
|---|---|
| map-editor.php・kanban.php・monitor.php の本文 | map-editor.php(埋め込み)・monitor.php とその API(health-check・chat-auth)を読んだ。**kanban.php は読んでいない** |
| map-edit.php の後半・app-map-convert.php の全体 | 取り込みの入口(大きさの上限・管理者だけ)を読んだ。**変換処理の全行は読んでいない** |
| 外部の CVE データベースとの照合 | **Composer は `composer audit` で照合した(0 件)。** コンテナイメージ(Trivy 等)は未照合 |
| Logto の中 | SDK(PHP 0.3.1)は読んだ。**Logto サーバーの設定(Console)は見ていない** |

### 2.2 前回より後に入った機能

| 機能 | 見たこと | 結果 |
|---|---|---|
| 教職員の組織ロール(docs/15、2026-09-18) | `lib/staff-org.php`・`lib/staff-nodes.php`・`account.php`・`admin/staff-*.php`・`logto_guard.php` の組織トークンの扱い | W-48・W-50・W-51 |
| 公開ページの教職員の印 | `index.php:56-61`・`lib/map-access.php` | W-50 |
| 地図のパスワードの錠 | `api/map-unlock.php`・`lib/map-access.php`・`admin/map-settings.php` | W-46 |

### 2.3 カタログの節ごとの結果(今回見たものだけ)

| カタログ | 結果 | 根拠・所見 |
|---|---|---|
| WA-01〜03 認可 | 問題なし | 管理画面は `guard.php` が全ページで判定。公開 API は `logto_guard.php` が iss・aud・client_id・停止・組織 ID を照合 |
| WA-02 IDOR | 問題なし | アバター・設定・ランキングの削除は、トークンの sub に対してだけ動く |
| WA-05 SSRF | 問題なし | health-check の宛先は固定 |
| WA-07 JWT | 問題なし | firebase/php-jwt 7.1 で JWKS 検証。`alg` は鍵の側で決まる |
| WA-19 依存 | 問題なし(Composer) | `composer audit` 0 件 |
| WA-34〜36 XSS | 問題なし | 地図・チャット・管理画面の出力はエスケープ済み。CSP に `'unsafe-inline'` なし |
| WA-44 レート制限 | **所見あり** | W-44・W-45(回数の上限はあるが、1 回に数えられる量に上限が無い) |
| WA-45 業務ロジック | **所見あり** | W-44(前回「未確認」だった「ランキングの水増し」) |
| WA-53〜54 ログ | **所見あり** | W-48・W-49(残し方と消し方の食い違い) |
| WA-62 資源枯渇 | **所見あり** | W-44(表の行が際限なく増える) |
| WA-28 秘密の直書き | 問題なし | このリポジトリの全履歴で 0 件 |
| CWE-200 情報の露出 | **所見あり** | W-47(公開リポジトリの IP) |
| LF-04 セッション固定 | 問題なし | 解除の時点で `session_regenerate_id(true)`(`api/map-unlock.php:107`)、`use_strict_mode=1` |
| LF-07 失効 | **所見あり** | W-46(パスワードを替えても解除が残る)・W-51 |
| LF-11 state / LF-13 PKCE | 問題なし | SDK が `state` を照合し、PKCE を使う |
| LG-07〜09 | 該当しない見込み | Logto 1.43.0。**ただしカタログの CVE 番号と修正版はこの診断で一次情報を確かめていない**(§6) |
| PM-01〜07 | 該当しない見込み | phpMyAdmin 5.2.3(同上) |
| SK-01〜04 | 問題なし | 既定値ではない鍵(`.env`)、presence だけ、クライアント送信なし、Origin 照合は前回 W-38 で済み |

## 3. 所見の一覧(W-44〜W-52)

| 番号 | 深刻度 | 判定 | 種類 | 場所 | 何か |
|---|---|---|---|---|---|
| W-44 | **中** | 実行で確認 | セキュリティ | `server/src/lib/app-ranking.php:111` | ランキングの記録が、1 回の送信に含まれる同じ地点の重複と実在しない地点 ID を数える。参加者は 1 回で自分の件数を最大 +50、誰でも表の行を際限なく増やせる |
| W-45 | 低 | 実行で確認 | セキュリティ | `server/src/lib/app-ranking.php:52` | 「調べられた語」の公開は「3 つ以上の送信元 IP」だけが条件で、1 人でも越えられる。任意の 64 文字を、誰でも見られる一覧に載せられる |
| W-46 | 低 | 実行で確認 | 個人情報 | `server/src/api/map-unlock.php:108` | 地図のパスワードを替えても、前のパスワードで解除済みのセッションは教職員氏名を見続けられる |
| W-47 | 低 | 実行で確認 | 個人情報 | `server/docs/12-hardening-2026-09-15.ipynb:168` | 公開リポジトリに運用者の回線の IP アドレス(4 箇所)。push 前の検査に IP の検出が無い |
| W-48 | 低 | 実行で確認 | 個人情報 | `server/src/admin/staff-requests.php:48` | アカウントを削除しても、対象者の利用者 ID が監査ログ(detail)とチャットの発言に残る。ポリシー §8 と食い違う |
| W-49 | 低 | 実行で確認 | 個人情報 | `server/src/lib/admin-log.php:167` | IP アドレスを含む監査ログとパスワード解除の記録に、保存期限も自動削除も無い。**解除に成功した人の IP も残る** |
| W-50 | 低 | 実行で確認 | 不具合 | `server/src/index.php:60` | 公開ページは ID トークンを更新しないので、教職員の特典(氏名・閲覧不可の地点)が ID トークンの期限で消える |
| W-51 | 情報 | 実行で確認 | セキュリティ | `server/src/account.php:145` | 教職員かどうかをセッション内の古い ID トークン(期限を見ない)で判定している。期限切れのトークンから出した提案が、承認されて地図に入った |
| W-52 | 情報 | 実行で確認 | 個人情報 | `server/src/api/app-map.php:177` | 配信の「最新です」の判定が役割を見ない。権限を失った端末の地図ファイルに氏名が残る(アプリ側は Android の報告書の A-32) |
| W-53 | **中** | 実行で確認 | 個人情報 | `server/src/scripts/check.php:1283` | 公開リポジトリに、実在の教職員の氏名と部屋の組が 4 人分(`check.php`・`Main/app.js:1860`・`docs/plan.md:555`)。全履歴にも同じ 4 人 |

## 4. 所見の詳細と、確かめるセル

### W-44 ランキングの記録が重複と実在しない地点を数える(中・実行で確認)

**カタログ:** WA-45 業務ロジックの悪用(CWE-840)、WA-62 資源枯渇(CWE-400)。

**どこ:**

| 行 | 中身 |
|---|---|
| `server/src/lib/app-ranking.php:178` `km_ranking_clean_uuids()` | 形(`[A-Za-z0-9_-]{1,64}`)と件数(50)だけ見る。**重複を落とさない・実在を確かめない** |
| `server/src/lib/app-ranking.php:128` | 1 件ごとに `visits = visits + 1`。同じ ID が 50 個あれば +50 |
| `server/src/lib/app-ranking.php:162` | 利用者の行は `visits + count($places)`。重複込みの件数がそのまま足される |
| `server/src/lib/app-ranking.php:149` | 新しい語は、送るたびに `km_map_ranking_queries` と `km_map_ranking_query_sources` に行が増える |
| `server/nginx/default.conf.template:237` | 入口の制限は 1 つの IP から毎分 120 回(burst 60) |

**起きること:**

1. ログインしてランキングに参加した人は、同じ地点 ID を 50 個並べて送るだけで、1 回につき自分の件数を +50 できる。
   毎分 120 回なら毎分最大 6,000。**利用者ランキングの 1 位を簡単に取れる**
2. ログイン不要で、「よく行かれた場所」の件数を同じやり方で水増しできる
3. **実在しない ID と新しい語は、送るたびに新しい行になる。** 1 つの IP から、場所の表に毎分最大 6,000 行、語の表とその送信元の表にも同じだけ行が増える。
   消えるのは年が 2 つ変わったときだけ(`KM_RANKING_RETENTION_YEARS = 2`)なので、**ディスクを埋められる**

アプリ側はもともと重複を送らない(`MapActivityTracker.kt` の `addVisit` が同じ ID を足さない)。**サーバーが同じ前提を確かめていない**のが原因。

**手元で動かした結果(2026-09-25):** 同じ ID を 60 個渡す → 50 個が残り、異なる値は 1 つ。実在しない ID と任意の語もそのまま通った。

In [ ]:
# 🟢 W-44 / W-45 を手元の PHP で確かめる(DB には繋がない。入力を絞る関数だけ動かす)
import subprocess
php = r"""
require 'lib/app-ranking.php';
$r = km_ranking_clean_uuids(array_fill(0, 60, '2735e277-ed55-4b74-963e-82baa0b6a641'));
echo '同じ ID を 60 個 → 残る件数 ', count($r), ' / 異なる値 ', count(array_unique($r)), PHP_EOL;
echo '実在しない ID → ', json_encode(km_ranking_clean_uuids(['not-a-real-node-000001'])), PHP_EOL;
echo '任意の語 → ', json_encode(km_ranking_clean_queries(['任意の文言']), JSON_UNESCAPED_UNICODE), PHP_EOL;
"""
print(subprocess.run(['php', '-r', php], cwd=km_nb.SERVER / 'src', capture_output=True, text=True, encoding='utf-8').stdout)

### W-45 「調べられた語」を 1 人で公開一覧に載せられる(低・実行で確認)

**カタログ:** WA-45(CWE-840)。載せられるのは文字列だけで、画面ではエスケープされるのでスクリプトは動かない(XSS ではない)。

**どこ:** `server/src/lib/app-ranking.php:52`(`KM_RANKING_QUERY_MIN_SOURCES = 3`)と `:304` `km_ranking_queries()`。
出どころは IP の鍵付きハッシュ(`:147`)。**IP を 3 つ用意すれば条件を満たす**(携帯回線の繋ぎ直し・自宅回線・VPN など)。
語の中身は長さ(64 文字)と制御文字しか見ない(`:204`)。

**起きること:** 誰でも見られる「調べられた語」の一覧(アプリのランキング画面・`GET /api/app-ranking.php`)に、
**特定の人を名指しする文言や不適切な文言を載せられる。** 一覧から消すには DB を直接触るしかない(管理画面に消す操作は無い)。

### W-46 地図のパスワードを替えても、前の解除が残る(低・実行で確認)

**カタログ:** LF-07 ログアウト時のトークン非失効(CWE-613)に近い。守っているものは**教職員氏名(個人情報)**。

**どこ:**

| 行 | 中身 |
|---|---|
| `server/src/api/map-unlock.php:108` | 解除に成功すると `$_SESSION['km_map_unlocked'] = true`。**どのパスワードで解いたかを持たない** |
| `server/src/lib/map-access.php:199` `km_map_password_entered()` | 印が true かだけを見る |
| `server/src/admin/map-settings.php:41` | パスワードを替えるのは設定ファイルのハッシュだけ。既存のセッションには触れない |

**起きること:** パスワードが漏れたので替えた、という場面で、**漏れたパスワードで既に解除した人は氏名を見続けられる。**
印が消えるのはサインアウト(`sign-out.php:118`)か、セッションが消えたときだけ。
PHP の既定(`session.gc_maxlifetime` = 1440 秒。`docker/php/99-limits.ini` は変えていない)では、
**最後の操作から 24 分以上**あいたセッションだけが、確率的に(既定 1/100 の割合で起きる掃除のときに)消える。使い続けている人の印は残る。

### W-47 公開リポジトリに運用者の回線の IP アドレス(低・実行で確認)

**カタログ:** CWE-200 情報の露出。WA-28(秘密の直書き)の検査の漏れでもある。

**どこ:** `server/docs/12-hardening-2026-09-15.ipynb` の **168・1391・1410・1853 行目**(JSON の行番号)。
同じアドレスが、このリポジトリの**過去のコミットにも残っている**(消しても履歴から読める)。

**なぜ問題か:**

- `server/nginx/km/allow-admin.conf` 自身が「**リポジトリに個人の所在情報を残さない**ため、自宅などの固定 IP は `allow-admin-home.local.conf` に置く」と決めている。その決まりと食い違う
- 文書によれば、このアドレスは**管理用ポート(phpMyAdmin・Logto Console・Mailpit)への接続を許した運用者の回線**で、**`kmops` の鍵もこの回線からしか使えない**(ホスト自身と 127.0.0.1 を除く)。狙う先を外に教えている
- 回線の IP は、おおよその所在地(と契約している事業者)に結び付く

**検査が止めなかった理由:** `tools/push-github.ps1:142-146` の検査は、パスワード・秘密鍵・トークン・一部のメールアドレスの形だけを見ていて、**IP アドレスを見ていない。**

**確かめるセル**(値は伏せて、どこにあるかだけ出す):

In [ ]:
# 🟢 W-47 公開される docs/ の中の IPv4 アドレスを探す(値は伏せる)
# 除くもの: 私用・文書用の範囲、公開 DNS、nginx/km/allow-admin.conf が allow しているホスト自身の住所、
# 版番号(jdk-21.0.12.101 のように前後が英字や - で続くもの)
import ipaddress, re
allowed_nets = [ipaddress.ip_network(n) for n in (
    '10.0.0.0/8', '172.16.0.0/12', '192.168.0.0/16', '127.0.0.0/8', '0.0.0.0/8',
    '192.0.2.0/24', '198.51.100.0/24', '203.0.113.0/24',   # 文書用(RFC 5737)
    '1.1.1.1/32', '1.0.0.1/32', '8.8.8.8/32', '8.8.4.4/32',  # 公開 DNS
)]
allow_conf = (km_nb.SERVER / 'nginx' / 'km' / 'allow-admin.conf').read_text(encoding='utf-8')
allowed_hosts = {ipaddress.ip_address(a) for a in re.findall(r'^allow\s+([0-9.]+);', allow_conf, re.M)}
pat = re.compile(r'(?<![\w.-])(\d{1,3}(?:\.\d{1,3}){3})(?![\w.-]*[A-Za-z])(?![\d.])')
for path in sorted([*km_nb.DOCS.glob('*.ipynb'), *km_nb.DOCS.glob('*.md')]):
    for no, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        for m in pat.finditer(line):
            try:
                ip = ipaddress.ip_address(m.group(1))
            except ValueError:
                continue
            if ip in allowed_hosts or any(ip in n for n in allowed_nets):
                continue
            masked = '.'.join(m.group(1).split('.')[:1] + ['*'] * 3)
            print(f'{path.name}:{no}  {masked}')

2026-09-25 の作業ツリーでは **`12-hardening-2026-09-15.ipynb` の 4 行(168・1391・1410・1853)だけが出る。** それが W-47 の対象。
ホスト自身の公開アドレス(`allow-admin.conf` の allow。DNS で誰でも引ける)と公開 DNS・版番号は除いてある。

### W-48 アカウントを削除しても、対象者の利用者 ID が残る(低・実行で確認)

**カタログ:** WA-54 ログへの機微情報(CWE-532)。**プライバシーポリシーとの食い違い。**

**ポリシー(`server/src/lib/legal.php:370`):**「管理操作の記録だけは残ります。ただし名前・利用者 ID・接続元 IP アドレス・
端末の名乗り(User-Agent)は削除するため、その記録から個人を特定することはできません。」

**実際(`server/src/lib/account-delete.php:153`):** 匿名にするのは `WHERE actor_id = 本人` の行、つまり**本人が操作した行だけ**。
管理者が**その人に対して**した操作の行は、実行者が管理者なので対象外。その `detail` に対象者の利用者 ID が入っている:

| 行 | 記録 | detail |
|---|---|---|
| `server/src/admin/staff-requests.php:48` | `staff.approved` / `staff.rejected` / `staff.revoked` | `#申請番号 利用者ID` |
| `server/src/admin/staff-nodes.php:47` | `staffnode.assigned` | `地点ID 利用者ID` |
| `server/src/admin/staff-nodes.php:53` | `staffnode.unassigned` | `地点ID 利用者ID` |

**チャットも同じ:** `km_chat_messages`(`server/src/lib/chat.php:16`)の `sender_id`・`sender_name` は、
削除する表の一覧(`account-delete.php:59` `KM_ACCOUNT_DELETE_TABLES`)に入っていない(列名が `user_id` ではないため、入れても今の DELETE 文では消えない)。
チャットは管理者どうしのものだが、**管理者が自分のアカウントを消しても、名前と ID が残る。** ポリシーはチャットの発言について何も書いていない。

**起きること:** 教職員が申請して承認され、あとでアカウントを消しても、「この利用者 ID が教職員として承認された」という行が監査ログに残る。
Logto からは消えているので ID から名前は引けないが、**ポリシーは「利用者 ID を削除する」と約束している。**

In [ ]:
# 🟢 W-48 監査ログの detail に利用者 ID を入れている箇所と、削除の対象表を並べる
import re
src = km_nb.SERVER / 'src'
for rel in ('admin/staff-requests.php', 'admin/staff-nodes.php'):
    for no, line in enumerate((src / rel).read_text(encoding='utf-8').splitlines(), 1):
        if re.search(r"\$detail = .*userId|km_admin_log_record\(.*\$userId", line):
            print(f'{rel}:{no}  {line.strip()}')
text = (src / 'lib/account-delete.php').read_text(encoding='utf-8')
tables = re.search(r'KM_ACCOUNT_DELETE_TABLES = \[(.*?)\];', text, re.S).group(1)
print('削除する表:', re.findall(r"'([a-z_]+)'", tables))
print('km_chat_messages は対象か:', 'km_chat_messages' in tables)

### W-49 IP アドレスの記録に保存期限が無い(低・実行で確認)

**カタログ:** WA-54 / DB-17 の近く(保存の最小化)。個人情報保護法の個人データに当たるかは、ほかの情報と照合できるかで変わるので、ここでは判断しない。

| 表 | 何が残るか | 消えるとき |
|---|---|---|
| `km_admin_log`(`server/src/lib/admin-log.php:167`) | 管理操作・**公開ページのパスワード失敗(`map.unlock_failed`)**・webhook の失敗の、接続元 IP(`:174`) | アカウント削除で本人の行だけ匿名化。**それ以外は消えない** |
| 失敗回数の表(`server/src/lib/map-rate-limit.php:98`) | **解除を試した IP(主キー)。成功した人も含む** —— 照合の前に必ず 1 回数える(`api/map-unlock.php` の `km_map_unlock_attempt`)ので、1 回で正しく解除した人も「失敗 0 回」の行として残る(§7 で確認) | **消えない**(失敗回数を 0 に戻すのは UPDATE。`:49` に理由がある) |

ポリシー(`server/src/lib/legal.php:349`)は「障害調査と不正アクセスへの対処に**必要な期間**保存します」とだけ書いている。
**その期間を決めて消す仕組みがコードに無い。** 公開ページで 1 回パスワードを間違えた来場者の IP が、ずっと残る。

### W-50 公開ページで、教職員の特典が ID トークンの期限で消える(低・実行で確認・不具合)

**どこ:**

| 行 | 中身 |
|---|---|
| `server/src/index.php:38` | セッションに入っている ID トークンを**読むだけ**(`getIdTokenClaims()`)。更新しない |
| `server/src/index.php:60` | 教職員の印の期限を `min(ID トークンの exp, 今 + 3600)` にする |
| Logto PHP SDK 0.3.1 `LogtoClient.php:76` | `isAuthenticated()` は **ID トークンがあるかだけ**を見る(期限は見ない) |
| SDK `LogtoClient.php:104` `:434` | ID トークンが新しくなるのは、`getAccessToken()` がリフレッシュしたときだけ |

**起きること:** サインインしてから **ID トークンの有効期間(Logto の設定値)** が過ぎると、`exp` が過去になる。
公開ページはリフレッシュする呼び出しを持たないので、印が立たなくなり、**教職員に氏名と閲覧不可の地点が出なくなる。**
画面の右上には「○○ さん」と出たまま(ID トークン自体は残っている)なので、利用者には理由が分からない。
戻るのは、サインインし直すか、`getAccessToken()` を呼ぶ画面(管理画面・`account.php` の保存)を開いたとき。

**検証(§7):** ID トークンの期限が 1 分前に切れた教職員のセッションで公開ページを開くと、右上には「○○ さん」と出たまま、氏名が出なくなった。
そのあいだ Logto へのトークンの更新の要求は 0 回で、ID トークンは古いまま。**起きるまでの時間は本番の Logto の ID トークンの有効期間で決まる(その値は見ていない)。**

### W-51 教職員の判定に、期限を見ない古い ID トークンを使う(情報・実行で確認)

**どこ:** `server/src/account.php:145`(地点の変更の提案)と `:311`(担当地点の表示)。
どちらも ID トークンの `organization_roles` を見るが、**その ID トークンの期限は見ない**(W-50 の SDK の性質)。
コメント(`:138-140`)は「申請の表だけで決めると、取り消したあとも続くセッションから送れてしまう」としているが、**ID トークンも同じ理由で古いまま残る。**

**実害が小さい理由:** 提案を承認するときに「申請が承認済みか」を表で見直す(`lib/staff-nodes.php` の `km_staff_node_edit_decide`)。
管理画面の「取り消し」を使えば、古い ID トークンからの提案は承認できない。

**残る穴:** Logto Console で**組織から直接外した**場合は、申請の表が `approved` のまま残る。
その人は古いセッションから、担当地点の今の値(教職員氏名を含む)を見て、承認されうる提案を送り続けられる。
**検証(§7):** 2 時間前に期限の切れた ID トークンのセッションで、アカウント画面に担当地点の氏名が出て、提案が受け付けられ、管理者が承認すると地図に入った。
**ただし** POST の経路はトークンを更新しないが、GET でアカウント画面を開くと更新が走る(そのとき本物の Logto が組織を外した ID トークンを返せば、次からは通らない)。この部分は偽の Logto では確かめていない。

### W-52 権限を失った端末の地図ファイルに氏名が残る(情報・実行で確認)

**どこ:** `server/src/api/app-map.php:177` の「最新です」の判定は、配信 ID と版だけを見る。**役割(来場者・教職員・スタッフ)を見ない。**
教職員の権限を取り消されても、アカウントを止められても、版が変わらない限り本体を送り直さない。

**実害が小さい理由:** アプリの画面は役割で氏名を隠す。見えるのは端末の中のファイル(アプリ専用領域・バックアップ対象外)だけ。
**それでも「権限が無くなったのに個人情報を持ち続ける」のは最小化の考え方に合わない。**

アプリ側(ログインの復元に失敗したときに地図ファイルから氏名を落とさない)は、非公開の Android リポジトリ(`itotakusub/Ichinoseki_Kosen`)の `docs/security-review-2026-09-25.ipynb` の **A-32**。

### W-53 公開リポジトリに、実在の教職員の氏名と部屋の組(中・実行で確認)

**カタログ:** CWE-200 情報の露出・CWE-359 個人情報の露出。**この報告の中でいちばん先に手を付けるもの。**

検証の途中で、非公開の Android のリポジトリにある DB のダンプ(`php/Kosen_map.sql`。地点 604 件のうち 55 件に氏名、54 人)と、
このリポジトリを突き合わせた。**同じ氏名が 4 人分、このリポジトリに出ている**(値はここに書かない)。

| 場所 | 中身 | ダンプ上の担当部屋と同じ行に並ぶか |
|---|---|---|
| `server/src/scripts/check.php:1283` | 自己検査のデータ。`'n_new_10' => ['name' => '(部屋名と番号)', 'occupant_name' => '(氏名)']` | **並ぶ**(ダンプと同じ地点 ID・部屋) |
| `server/src/scripts/check.php:393` `:412` `:468` `:1071` `:1107` `:1129` `:1201` | 自己検査のデータ(氏名だけ) | 並ばない(同じ氏名) |
| `server/src/Main/app.js:1860` | コメント。「移行前の graph.js は "(部屋名と番号) (氏名)" のように1本の文字列だった」 | **並ぶ** |
| `server/docs/plan.md:555` | 作業の記録。「`(部屋名と番号)` に `担当: (氏名) 先生` が出た」 | **並ぶ** |

**なぜ問題か:** 教職員氏名は、地図のパスワード(`lib/map-access.php`)・イベント中の非表示・アプリの役割で
**わざわざ隠している情報。** それが、誰でも読める公開リポジトリに、部屋番号と組で置かれている。
過去のコミットにも同じ 4 人が残っている(全履歴で 4 人)。

**直し方の案:** テストのデータとコメントを架空の名前(例: 「架空 太郎」)に替える。履歴からも消すなら書き換えが要る(W-47 と同じ注意)。
あわせて push 前の検査(`tools/push-github.ps1`)に「地図の氏名の一覧と照合して止める」検査を足すと、同じことが起きにくい。

## 5. 直し方の案(利用者の判断待ち)

**どれもまだ直していない。** 直すと決めたものから手を付ける。

| # | 番号 | 案 | 注意 |
|---|---|---|---|
| 1 | W-44 | `km_ranking_clean_uuids()` で重複を落とし(`array_unique`)、`km_map_nodes` に在る UUID だけ数える。利用者の行は、1 回の送信で足せる件数に上限を置く | 実在の確認は 1 回の `SELECT … IN (…)` で済む |
| 2 | W-44 / W-45 | 語の表に、年ごとの行数の上限か、一定件数に満たない語を定期的に消す処理を足す。管理画面に「語を一覧から外す」操作を足す | 公開の条件(3 つの送信元)を上げると、正当な語も出にくくなる |
| 3 | W-46 | 解除の印に、解除したときのパスワードのハッシュ(の鍵付き要約)を入れ、`km_map_password_entered()` で今の設定と比べる | 替えた瞬間に、全員がもう一度パスワードを聞かれる |
| 4 | W-47 | 12 のノートブックの IP を「この PC の回線(`allow-admin-home.local.conf` に書いた値)」の書き方に替える。`push-github.ps1` に、私用・文書用以外の IPv4 を止める検査を足す(ホストの公開アドレスは許可リストへ) | **過去のコミットには残る。** 消すなら履歴の書き換え(公開リポジトリなので fork・キャッシュには残りうる)。回線の IP を変える方が確実な場合もある |
| 5 | W-48 | 対象者の ID を detail に書く記録も、削除のときに ID を伏せる(`detail LIKE '% 利用者ID'` を置き換える)か、detail には申請番号だけを書く。チャットは `sender_id` で消すか匿名化する | 監査の追いやすさとの兼ね合い。**ポリシーの文を直す**という選び方もある |
| 6 | W-49 | 期間を決めて(例: 監査ログの IP は 90 日、失敗回数の表は 30 日)、定期処理で IP の列を NULL にする・行を消す。ポリシーにその日数を書く | 日数は運用で決める |
| 7 | W-50 | `index.php` で `getAccessToken()` を 1 回呼ぶ(期限が近ければ ID トークンも更新される)か、`exp` が過去なら「サインインし直してください」と出す | 公開ページに Logto への往復が増える |
| 8 | W-51 | 教職員の判定の前に ID トークンの `exp` を見る。Console から直接外す運用をしないと決め、手順書に書く | — |
| 9 | W-52 | 本体を送るかの判定に「前回どの役割で受け取ったか」を足す(アプリが送る値を増やす) | Android の A-32 と一緒に直す |
| 10 | W-53 | `check.php`・`Main/app.js`・`docs/plan.md` の実名を架空の名前に替える。push 前の検査に氏名の照合を足す | **いちばん先に。** 履歴に残る点は W-47 と同じ |

### 5.1 直した(2026-09-25 同日)

**利用者の決定:** 公開リポジトリは履歴も書き換える・保存期限は 90 日 / 1 日・語は管理画面で外せるようにする。
**本番への配備は利用者が行う**(下の「本番で確かめること」まで済んで、本当に直ったと言える)。

| 番号 | 状態 | どう直したか | 確かめた試験 |
|---|---|---|---|
| W-44 | 直した | 1 回の送信の中の重複を落とす。地点は `km_map_nodes.uuid` に在るものだけ数える(確かめられなければ数えない)。利用者の件数は前の記録から 30 秒たっていなければ足さない。語の表は年 20,000 行まで、語ごとの出どころの印は 12 まで(`lib/app-ranking.php`) | check.php `review-0925`(SQLite で実在の絞り込みを動かす) |
| W-45 | 直した | 出どころを `km_map_rate_limit_key()`(IPv6 は /64)で数える。管理画面 **「ランキングの語」**(`admin/ranking.php`)で、公開前の語も含めて一覧から外せる。外した語はその年は数えない。記録は `ranking.query_hidden`(語そのものは書かない) | 同上 |
| W-46 | 直した | 解除の印に、そのときのパスワード設定の要約(`km_map_password_fingerprint`)を入れ、今の設定と比べる。**配備のあと、解除済みの人も一度だけ入れ直しになる** | check.php `map-access` |
| W-47 | 直した | 12 の IP を「この PC の回線の IP(値はリポジトリに書かない)」に替えた。`push-github.ps1` に公開の IPv4 の検査と `-ScanAll` を足した。**履歴の書き換えは force push の直前に利用者へ確かめる** | `push-github.ps1 -CheckOnly -ScanAll` で 0 件 |
| W-48 | 直した | 削除のとき、監査ログの detail の対象者 ID を「(削除された利用者)」に置き換え、チャットの発言者も同じにする(本文は残す)。ポリシーにも書いた | check.php `review-0925` |
| W-49 | 直した | `lib/privacy-retention.php`: 監査ログの IP と端末名は 90 日で空に、解除の試行の記録は 1 日で消す(1 日 1 回、記録のついでに動く)。成功した解除の行はその場で消す。ポリシーに日数を書いた(定数から出す) | 同上 |
| W-50 | 直した | `KmLogtoClient::kmFreshIdTokenClaims()` が、期限の近い ID トークンを refresh_token で取り直す。取り直せない教職員には「サインインし直してください」と出す | 偽の OidcCore で 5 通り |
| W-51 | 直した | account.php の 2 箇所も同じ関数で判定する。**運用: 取り消しは管理画面で行い、Console で組織から直接外さない**(docs/15) | 同上 |
| W-52 | 直した | 配信に中身の段(`visitor` / `staff` / `names`)を載せ、アプリが `haveLevel` で返す。段が違えば同じ版でも送り直す(古いアプリは今までどおり) | check.php・Android `ReviewFixes0925Test` |
| W-53 | 直した | 実名 6 人分(報告の 4 人+計画書の 1 行に並んでいた 2 人)を架空の名前に替えた。`push-github.ps1` がリポジトリの外の氏名の一覧(`update-private-names.ps1` が作る)と照合する。**履歴は W-47 と同じ** | 同上(0 件) |

**本番で確かめること**(配備のあと。利用者が行う):
W-44 同じ地点 ID を並べて送っても件数が 1 だけ増える/W-46 解除 → 管理画面でパスワードを替える → 同じブラウザで氏名が消える/
W-50 教職員で入り、ID トークンの期限を過ぎてから公開ページを開いても氏名が出る(**本物の Logto が取り直しで organization_roles を返すか**はここで分かる)/
W-49 翌日以降、解除の試行の表に古い行が無い/W-52 新しい APK で教職員として地図を取る → 権限を取り消す → 「更新」で氏名が消える。

## 6. この診断の限界

| 見ていないもの | どう困るか | 埋めるには |
|---|---|---|
| **カタログの 2026 年の CVE 番号・CERT/CC VU#492466・修正版の番号** | この診断では NVD・GitHub Advisory・CERT/CC の一次情報を確かめていない。**「Logto 1.43.0 なら LG-07〜09 は直っている」はカタログの記述が正しい前提** | Logto の公式リリースノートとセキュリティアドバイザリを読む |
| **コンテナイメージの既知の脆弱性** | Composer は照合したが、イメージ(OS のパッケージ)は照合していない | Trivy / Grype で |
| **本番** | 検証は手元の検証環境だけ(§7)。nginx・Apache・本物の Logto・Soketi・メールは入れていない。**本番には一切触れていない** | 直したあと、本番で同じ手順を 1 つずつ試す |
| **Logto の中(Console の設定)** | ID トークンの有効期間(W-50 が起きるまでの時間)、組織から直接外す運用の有無(W-51)。本物の Logto がリフレッシュで返す ID トークンの中身 | Console を人が見る |
| **読んでいないコード** | kanban.php・projects.php・calendar.php の本文、app-map-convert.php の変換処理の全体、scripts の ps1/sh の多く | 観点を決めて読む |
| **侵入試験(DAST)** | 実際に攻撃を投げていない | 使い捨ての環境で |
| **法的な判断** | W-48・W-49 が個人情報保護法の「個人データ」の扱いとしてどうか(照合の容易性)は判断していない | 学校の個人情報の担当に確かめる |

## 7. 検証(2026-09-25)

### 7.1 検証環境

**本番と同じもの:** MariaDB `11.4`(`compose.yaml` と同じ digest のイメージ)、このリポジトリの `server/src` の写し、
`composer.lock` どおりの依存(Logto PHP SDK 0.3.1・firebase/php-jwt 7.1.0)、DB の表定義(DB のダンプから CREATE/ALTER だけを取り出したもの)と移行 SQL。

**代わりのもの(本番と違う点):**

| 本番 | 検証 | 影響 |
|---|---|---|
| nginx → Apache + PHP 8.4 | PHP 8.4.19 の組み込みサーバー(`php -S`) | **nginx の回数制限と `X-Real-IP` の上書きが無い。** 送信元 IP は `X-Real-IP` を直に付けて装った(本番では nginx が本当の接続元で上書きする) |
| Logto | 偽の Logto(`mock/router.php`)。discovery・JWKS・client_credentials・利用者の取得・組織の出し入れに決まった応答を返し、受けた要求を記録する | ブラウザのセッションは、SDK がセッションに置くのと同じ形で作った(**SDK は保存済みのトークンを検証しない**ので、これで SDK の振る舞いはそのまま)。アプリのアクセストークンは偽の Logto の鍵で**本当に署名**し、`logto_guard.php` の検証を通した |
| Soketi・メール | 無し | チャットの配信とメールの知らせは失敗する(どちらも保存の後なので、検証に影響しない) |
| 地図のデータ | **架空の地点 1 件**(氏名「架空 太郎」) | — |

**正直に書いておくこと:** 表定義を入れるとき、はじめは取り出し方を誤って、ダンプの**データ**(実名を含む)まで手元の使い捨ての DB に読み込んだ。
気付いてすぐ全行を消し、架空のデータに置き換えた。やり直し用の `setup.sh` は表定義だけを使う(`schema.sql` に INSERT は 0 件)。
検証の DB は検証の後に消した。

### 7.2 やり直し方

このリポジトリの **`docs/verify-2026-09-25/`**(`server/` の外に置いた。`tools/sync-from-website.ps1` の /MIR で消えないように)。

```sh
cd docs/verify-2026-09-25
./setup.sh      # DB のコンテナ・コードの写し・依存・鍵・配信の設定を作る(docker・php 8.4・composer・openssl・python3 が要る)
./run_all.sh    # 所見を順に確かめ、results/ に残す
```

W-53 は、非公開の Android のリポジトリがこのリポジトリの**隣**(`../Ichinoseki_Kosen`)にあるときだけ走る。

### 7.3 所見ごとの結果

| 番号 | 何をしたか | 結果 |
|---|---|---|
| W-44 | 公開 API に同じ ID 50 個・実在しない ID 50 個×10 回を送った。参加者の分は、API が認証の後に呼ぶ関数を同じ引数で呼んだ | **成り立つ。** 1 回で件数 50。場所の表が 500 行、語の表と送信元の表も 500 行ずつ増えた(地図の地点は 1 件) |
| W-45 | 同じ語を送信元 IP を変えて 3 回送った | **成り立つ。** 3 つ目の IP の後、ログイン不要の一覧に載った |
| W-46 | パスワード A で解除 → 管理側の関数で B に変更 → 同じセッションで地図のデータを取る | **成り立つ。** 変更後も氏名が出た。新しいセッションでは A が拒否された |
| W-47 | `push-github.ps1` の正規表現を IP の行に当てた(pwsh が無いので Python で同じ式) | **成り立つ。** 4 行とも、どの検査にも当たらない |
| W-48 | 申請 → 承認 → 割り当て → チャット → 署名付きの `User.Deleted` を 2 人分 | **成り立つ。** detail の利用者 ID 2 行と、チャットの ID・名前が残った。申請と割り当ての行は消えた |
| W-49 | 解除の失敗と成功を 1 回ずつ | **成り立つ。しかも想定より広い**(成功した IP も行として残る)。消す処理はコードに無い |
| W-50 | ID トークンの期限が 1 時間先 / 1 分前の教職員で公開ページと地図のデータを取る | **成り立つ。** 期限切れでは氏名が出ず、トークンの更新は 0 回 |
| W-51 | 2 時間前に期限の切れた ID トークンで、アカウント画面・提案・承認 | **成り立つ。** 提案が承認されて地図に入った |
| W-52 | 本当に署名したスタッフのトークンで取得 → トークン無し / 権限無しのトークンで、手元の版を送って取得 | **成り立つ。** どちらも「最新です」で、氏名入りの地図は置き換わらない |
| W-53 | DB のダンプの「地点 → 氏名」と、このリポジトリの全ファイル・全履歴を突き合わせた(氏名は伏せて出す) | **成り立つ。** 4 人。うち 3 人は部屋番号と同じ行 |

### 7.4 出力(results/ の中身)

```text
== W-44 (1) 同じ地点 ID を 50 個並べて 1 回だけ送る(ログインなし)
{"success":true,"recorded":true}
   その地点の件数:
aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaa1	50
== W-44 (2) 実在しない ID を 50 個ずつ、10 回送る
   場所の表: 1 行 → 501 行
   語の表: 500 行 / 語の送信元の表: 500 行
   地図に在る地点の数: 1
== W-44 (3) ログインした参加者: API が認証の後に呼ぶ km_ranking_record() を、同じ引数の形で呼ぶ
   1 回の記録の後の利用者ランキング: [{"userId":"udeaecd98cc1e215b6cef7c1c4569966","displayName":"参加者A","visits":50}]
== W-45 任意の語を、送信元 IP を 1 つずつ増やして送る
   203.0.113.21 から送った後の公開一覧(GET): 載らない
   203.0.113.22 から送った後の公開一覧(GET): 載らない
   203.0.113.23 から送った後の公開一覧(GET): 載った

[設定] 氏名の錠 = password、パスワード = PassA
1) Cookie なし:
namesUnlocked=false occupantName=null
2) パスワード A で解除:
  応答: {"success":true}
  セッション: efdea3d0…
3) 解除したセッションで:
namesUnlocked=true occupantName="架空 太郎"
[設定] 氏名の錠 = password、パスワード = PassB
4) パスワードを B に替えた後、同じセッションで:
namesUnlocked=true occupantName="架空 太郎"
5) 新しいセッションで古いパスワード A を試す:
  応答: {"success":false,"message":"パスワードが正しくありません。"}
namesUnlocked=false occupantName=null

12-hardening-2026-09-15.ipynb:168 → 検査に当たったもの: なし(push が止まらない)
12-hardening-2026-09-15.ipynb:1391 → 検査に当たったもの: なし(push が止まらない)
12-hardening-2026-09-15.ipynb:1410 → 検査に当たったもの: なし(push が止まらない)
12-hardening-2026-09-15.ipynb:1853 → 検査に当たったもの: なし(push が止まらない)
IPv4 を探す検査があるか: False

1) 教職員が申請する(POST /account.php)
302 http://127.0.0.1:3900/account.php?staff=1
   申請番号 #1
2) 管理者が承認する(POST /admin/staff-requests.php)
302 http://127.0.0.1:3900/admin/staff-requests.php?done=approve
3) 管理者が地点を割り当てる(POST /admin/staff-nodes.php)
302 http://127.0.0.1:3900/admin/staff-nodes.php?done=assign
4) 管理者がチャットで発言する(POST /admin/api/chat-send.php。Soketi が居ないので 503 だが、保存は配信より先)
503 
   削除の前: 監査ログで teacher-0001 を含む行(実行者, 操作, detail)
teacher-0001	staff.requested	#1
admin-0001	staff.approved	#1 teacher-0001
admin-0001	staffnode.assigned	aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa1 teacher-0001
5) Logto から教職員と管理者の User.Deleted が届く(署名付き webhook)
{"ok":true,"message":"deleted"}
{"ok":true,"message":"deleted"}
== 削除の後
   監査ログで teacher-0001 を含む行(実行者, 操作, detail):
(NULL)	staff.approved	#1 teacher-0001
(NULL)	staffnode.assigned	aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa1 teacher-0001
   申請・割り当ての表に残る行: 0
   チャットに残る管理者 A の発言(sender_id, sender_name, message):
admin-0001	管理者A	検証用の発言
   監査ログで admin-0001 が実行者の行: 0(匿名化されていれば 0)

1) 失敗の記録(監査ログ): action, IP
map.unlock_failed	198.51.100.2
2) 失敗回数の表: IP, 回数
198.51.100.1	0
198.51.100.2	1
3) 同じ IP から正しいパスワード(B)で成功させる
{"success":true}
4) 成功の後も失敗回数の表に行が残るか: IP, 回数
198.51.100.1	0
198.51.100.2	0
5) これらの表を消す(DELETE・TRUNCATE・期限つきの掃除)処理がコードにあるか:
websrc/lib/map-events.php:430:            $pdo->prepare("DELETE FROM {$table} WHERE event_id = ?")->execute([$id]);
websrc/lib/app-ranking.php:256:        $pdo->prepare("DELETE FROM {$table} WHERE year < ?")->execute([$oldest]);
websrc/lib/account-delete.php:114:                $stmt = $pdo->prepare("DELETE FROM {$table} WHERE user_id = ?");
websrc/lib/account-delete.php:134:            $stmt = $pdo->prepare('DELETE FROM km_admin_log WHERE actor_id = ?');
websrc/scripts/reset-app-ranking.php:98:        $pdo->exec("DELETE FROM {$table}");

== W-50 (比較) ID トークンの期限が 1 時間先の教職員
   画面の表示: 👤 教職員teacher-f さん
   教職員の印の期限: {"km_map_teacher_until":1790305592}  (今は 1790301992)
   namesUnlocked=true occupantName="架空 太郎"
== W-50 ID トークンの期限が 1 分前に切れた教職員(リフレッシュトークンは持っている)
   画面の表示: 👤 教職員teacher-s さん
   教職員の印の期限: {"km_map_teacher_until":1790301932}  (今は 1790301992)
   namesUnlocked=false occupantName=null
   偽 Logto が受けた要求のうち、トークンの更新(grant_type=refresh_token)の数: 0
   ID トークンは変わったか: exp=1790301932(変わっていない)

== W-51 準備: 教職員 teacher-0002 を申請・承認・地点の割り当てまで済ませる
   申請: 302 http://127.0.0.1:3900/account.php?staff=1
   承認: 302 http://127.0.0.1:3900/admin/staff-requests.php?done=approve
   割り当て: 302 http://127.0.0.1:3900/admin/staff-nodes.php?done=assign
== W-51 この教職員の ID トークンを「2 時間前に切れた」ものに差し替える(Console で組織から外した後も、セッションに残る古いトークンを想定)
   アカウント画面に担当地点の現在の氏名が出るか: 出る
   地点の変更を提案(POST /account.php do_account=staff_node_edit): 302 http://127.0.0.1:3900/account.php?node=sent
   提案の表(利用者, 状態, 内容):
teacher-0002	pending	{"occupantName":{"from":"架空 太郎","to":"期限切れのトークンから"}}
   申請の表の状態: approved
   管理者が提案 #1 を承認: 302 http://127.0.0.1:3900/admin/staff-nodes.php?done=approve
   地図の地点の氏名: 期限切れのトークンから

1) スタッフのトークンで初めて取得(手元に地図なし):
   本体を受け取った revision=1 occupantName="架空 太郎"
2) トークン無し(= ログアウト・失効・停止の後)で、手元の版を送って取得:
   upToDate=true(本体を送らない) revision=1
3) 権限の無いトークン(= スタッフの権限を外された後)で、手元の版を送って取得:
   upToDate=true(本体を送らない) revision=1
4) 比べるため、トークン無しで手元の版を送らずに取得:
   本体を受け取った revision=1 occupantName=null

ダンプ: 地点 604 件、氏名のある地点 55 件、異なる氏名 54 人
公開リポジトリ(HEAD)に出る氏名: 4 人
  氏名#3  server/docs/plan.md:555  同じ行に、ダンプ上の担当部屋の番号もある
  氏名#3  server/src/scripts/check.php:393  同じ行に、ダンプ上の担当部屋の番号も無い
  氏名#3  server/src/scripts/check.php:468  同じ行に、ダンプ上の担当部屋の番号も無い
  氏名#8  server/src/Main/app.js:1860  同じ行に、ダンプ上の担当部屋の番号もある
  氏名#19  server/src/scripts/check.php:1071  同じ行に、ダンプ上の担当部屋の番号も無い
  氏名#19  server/src/scripts/check.php:1107  同じ行に、ダンプ上の担当部屋の番号も無い
  氏名#19  server/src/scripts/check.php:1129  同じ行に、ダンプ上の担当部屋の番号も無い
  氏名#19  server/src/scripts/check.php:1201  同じ行に、ダンプ上の担当部屋の番号も無い
  氏名#19  server/src/scripts/check.php:1283  同じ行に、ダンプ上の担当部屋の番号もある
  氏名#26  server/src/scripts/check.php:412  同じ行に、ダンプ上の担当部屋の番号も無い
公開リポジトリの全履歴に出る氏名: 4 人
```